# 第七课：模型微调 —— 打造属于你的AI专家

## 学习目标
- 理解微调的概念：在通用模型上进行「专业进修」
- 通过代码模拟微调前后的效果对比
- 理解微调数据的格式与质量要求
- 掌握「什么时候该微调、什么时候不该微调」的判断框架

> 微调让通用 AI 变成你的领域专家。但微调不是万能的——知道什么时候用才是关键。

## 环境准备

> 请先运行 `00_Environment_Setup.ipynb` 完成环境配置（安装依赖包 + 设置 API Key），
> 然后再回到本 Notebook。

完成后，运行下面的代码加载环境变量：

In [ ]:
# 从 .env 文件加载 API Key（无需每次输入）
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ===== 选一个服务商：只改这一行，其他都不用动 =====
#   'openai'     云端  需要 OPENAI_API_KEY      效果最强，支持 Embedding
#   'deepseek'   云端  需要 DEEPSEEK_API_KEY    云端最便宜，无 Embedding
#   'openrouter' 云端  需要 OPENROUTER_API_KEY  可调用多家模型，无 Embedding
#   'ollama'     本地  不需要 Key，免费离线     先跑 `ollama serve` 并 pull 模型
PROVIDER = 'openai'

# 下面四家都兼容 OpenAI 的接口格式，区别只在：地址、Key、模型名。
PROVIDERS = {
    'openai': {
        'base_url': None,                            # None = 用 OpenAI 官方默认地址
        'api_key': os.getenv('OPENAI_API_KEY'),
        'model': 'gpt-5.6-luna',                     # 小模型：便宜、快
        'model_big': 'gpt-5.6-terra',                # 大模型：贵、强
        'embedding_model': 'text-embedding-3-small',
    },
    'deepseek': {
        'base_url': 'https://api.deepseek.com/v1',
        'api_key': os.getenv('DEEPSEEK_API_KEY'),
        'model': 'deepseek-v4-flash',                # 快、便宜
        'model_big': 'deepseek-v4-pro',              # 更强、更慢；V4 两个模型都会先思考再回答
        'embedding_model': None,                     # DeepSeek 目前不提供 Embedding 接口
    },
    'openrouter': {
        'base_url': 'https://openrouter.ai/api/v1',
        'api_key': os.getenv('OPENROUTER_API_KEY'),
        'model': 'openai/gpt-5.6-luna',
        'model_big': 'openai/gpt-5.6-terra',
        'embedding_model': None,                     # OpenRouter 不转发 Embedding 接口
    },
    'ollama': {
        'base_url': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1'),
        'api_key': 'ollama',                         # 本地模型不校验 Key
        'model': 'gemma4:e2b-mlx',                   # 需先 ollama pull gemma4:e2b-mlx
        'model_big': 'gemma4:e2b-mlx',
        'embedding_model': 'nomic-embed-text',       # 需先 ollama pull nomic-embed-text
    },
}

cfg = PROVIDERS[PROVIDER]

# 先检查 Key：Key 为空时 OpenAI 客户端会直接抛出一长串报错，不容易看懂。
if not cfg['api_key']:
    raise SystemExit(
        f"没读到 '{PROVIDER}' 的 API Key。请在 .env 文件里补上 {PROVIDER.upper()}_API_KEY，\n"
        f"或者把上面的 PROVIDER 改成 'ollama'，用本地模型运行，完全不需要 Key。"
    )

client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])

# 后面所有代码都只用这三个变量，换服务商不需要改任何一行业务代码
MODEL = cfg['model']
MODEL_BIG = cfg['model_big']
EMBEDDING_MODEL = cfg['embedding_model']

print(f'连接成功！服务商 = {PROVIDER}，默认模型 = {MODEL}')


---

## 活动一：感受「通用模型」和「模拟微调后模型」的差异

### 活动目标
真正的微调需要大量 GPU 和时间，不适合课堂演示。但我们用 system prompt 来模拟微调效果——
给 AI 注入「专业知识」，观察它从「泛泛而谈」到「专业回答」的变化。

这本质上是提示工程，但它能帮你直观理解「微调」的目标：让 AI 在特定领域表现得像专家。

In [ ]:
# 活动一：模拟微调效果对比

# 一个非常专业的问题
question = '在RAG系统中，Chunk Size（文档分块大小）和Top-K（检索数量）这两个参数应该如何权衡？'

# 场景一：通用模型（没有领域知识注入）
print('=== 场景一：通用模型（没有领域知识）===')
r1 = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':question}],
    temperature=0.3)
generic_answer = r1.choices[0].message.content
print(generic_answer)

# 场景二：「模拟微调」——注入领域专业知识
print('\n=== 场景二：「模拟微调」——注入专业知识 ===')
domain_knowledge = (
    '你是RAG系统设计专家，拥有以下专业知识：\n'
    '1. Chunk Size 越小，检索精度越高但可能丢失上下文；建议范围256-1024 tokens\n'
    '2. Top-K 越大，召回率越高但引入噪声越多；建议范围3-10\n'
    '3. 经验法则：回答事实性问题用小Chunk+大Top-K；推理性问题用大Chunk+小Top-K\n'
    '4. 推荐起点：Chunk Size=512，Top-K=5，然后根据实际效果调整\n'
    '5. 如果用户反馈回答不够完整，增大Top-K；如果回答冗长混乱，减小Top-K'
)
r2 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':domain_knowledge},
        {'role':'user','content':question}
    ],
    temperature=0.3)
fine_tuned_answer = r2.choices[0].message.content
print(fine_tuned_answer)

# 对比两个回答
print('\n=== AI 裁判对比 ===')
comparison = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':
        f'请对比以下两个关于RAG参数的回答，指出哪个更专业、更有深度：\n\n'
        f'回答A（通用模型）：{generic_answer}\n\n回答B（注入专业知识后）：{fine_tuned_answer}'
    }],
    temperature=0.2)
print(comparison.choices[0].message.content)

### 讨论
- 注入专业知识后，AI 的回答发生了什么变化？
- 这个「模拟微调」和真正的微调有什么本质区别？
  - 提示工程：每次都要在 prompt 中重复注入知识
  - 真正的微调：知识已经融入模型参数中，不需要每次都写

---

## 活动二：微调 vs RAG vs 提示工程 —— 三者对比

### 活动目标
三种让 AI 适配特定场景的方法各有优劣。通过一个具体场景，对比它们的适用性。

In [ ]:
# 活动二：三种适配方法的对比

scenario = '你是一家律师事务所，需要一个AI助手来回答客户关于劳动法的常见问题。'

comparison_prompt = f'''场景：{scenario}

请从以下三个维度分析每种方法的适用性：
1. 准确性：答案的可靠性
2. 成本：实施和维护的成本
3. 灵活性：当法规变化时，更新是否容易

请用表格形式输出对比结果。'''

print('=== 三种方法对比分析 ===')
r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'你是一位AI工程化顾问。'},
        {'role':'user','content':comparison_prompt}
    ],
    temperature=0.3)
print(r.choices[0].message.content)

# 追问：推荐哪种方案？
print('\n=== 推荐方案 ===')
r2 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'user','content':f'针对{scenario}，你会推荐哪种方案？请给出具体理由和实施方案。'}
    ],
    temperature=0.3)
print(r2.choices[0].message.content)

### 讨论
- 三种方法中，哪种最适合「劳动法问答」这个场景？
- 什么情况下微调是最好的选择？什么情况下 RAG 更好？
- 「微调和 RAG 可以结合使用」——你能想出一个结合的场景吗？

---

## 活动三：设计一份微调数据集

### 活动目标
微调需要「训练数据」——一组高质量的问答对。这个活动让你体验数据准备的过程。
大多数人以为微调最难的是技术，其实是数据。

In [ ]:
# 活动三：设计微调数据集

# 场景：微调一个「咖啡店客服AI」
scenario_desc = ('你是一家连锁咖啡店的运营经理，需要微调一个客服AI。'
                'AI需要能够回答关于菜单、营业时间、会员积分、退换货等问题。')

print('场景：', scenario_desc)

# 让AI帮你生成训练数据样例
data_gen_prompt = f'''{scenario_desc}

请生成10对「客户问题 → 理想客服回答」的训练数据，覆盖以下场景：
1. 菜单咨询（2对）
2. 营业时间（2对）
3. 会员积分（2对）
4. 投诉处理（2对）
5. 特殊需求（2对）

要求：
- 回答要符合品牌调性：温暖、专业、高效
- 回答要包含具体信息（不要模糊回应）
- 格式：问题: ...\n回答: ...'''

r = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':data_gen_prompt}],
    temperature=0.7)
print(r.choices[0].message.content)

print('\n---')
print('检查要点：')
print('1. 这些问题和回答覆盖了你想要的所有场景吗？')
print('2. 回答的语气是否符合品牌调性？')
print('3. 有没有模糊、敷衍的回答需要修改？')
print('4. 10对数据远远不够——真正的微调通常需要数百到数千对！')

### 讨论
- AI 生成的数据质量如何？有没有需要人工修正的地方？
- 如果这些数据中有偏见或错误，微调后的模型会怎样？
- 数据质量 > 数据数量——你同意这个说法吗？

---

## 本节回顾

| 技能 | 说明 |
|------|------|
| 微调概念 | 理解微调是让通用模型获得专业能力的手段 |
| 三种方法对比 | 理解微调/RAG/提示工程各自的适用场景 |
| 数据集设计 | 体验微调数据准备的流程和质量要求 |

### 课后练习
1. 访问 OpenAI Platform 的 Fine-tuning 页面，了解微调的定价和流程
2. 搜索「LoRA fine-tuning explained simply」，了解参数高效微调
3. 思考：如果你要微调一个 AI，你会选择什么场景？需要准备多少数据？